<a href="https://colab.research.google.com/github/xyt556/I-GUIDE-GeoAI-Education/blob/main/notebooks/10-pixel-regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 像素级回归

## 介绍

许多重要的地球科学量，如冠层高度、生物量、土壤湿度和植被指数，是连续变量而不是类别。像素级回归通过使用类似于分割的编码器-解码器架构，但具有不同的输出层、损失函数和评估指标，为每个像素预测一个连续值。

本教程将通过`geoai`包，从数据准备到训练、评估和推理，逐步讲解完整的回归工作流程。本案例研究使用Landsat图像与NDVI数据配对，以预测植被指数值。

分类告诉您存在什么，分割告诉您它在哪里，而回归告诉您每个位置存在多少可测量的量。这种定量输出可直接用于科学分析和政策决策，而不会因离散化连续变量而丢失信息。

## 学习目标

通过本教程，您将能够：

- 解释像素级分类和像素级回归之间的区别
- 描述遥感图像连续值预测的实际应用
- 调整编码器-解码器分割架构以适应回归任务
- 使用`geoai.create_regression_tiles`准备配对的图像和目标栅格
- 使用`geoai.train_pixel_regressor`训练像素级回归模型
- 使用RMSE、MAE、R平方和残差分析评估回归结果
- 使用重叠瓦片和混合对大型栅格进行推理
- 将训练好的模型应用于新图像进行时间预测

## 理解像素回归

### 分类与回归

在分类中，模型从预定义的离散标签集中选择，并使用交叉熵损失。回归预测一个连续的数值（例如NDVI为0.65或冠层高度为12.3米），并使用MSE或其变体，惩罚预测值和实际值之间差异的大小。

这种区别在整个管道中具有实际意义：

- **输出层**：分类使用N个类别的softmax激活。回归使用单个输出通道，没有激活（或ReLU以强制非负值）。
- **损失函数**：交叉熵变为MSE、MAE或Huber损失。
- **评估指标**：准确性和IoU变为RMSE、MAE和R平方。
- **标签格式**：整数类别掩码变为具有连续值的浮点栅格。

### 应用

像素级回归在地理空间领域具有广泛的适用性：

**植被指数预测。** 可以从多光谱图像预测NDVI和其他光谱指数，从而实现时间序列填补和跨传感器协调。

**冠层高度估计。** 在LiDAR衍生高度图上训练的模型可以推广到LiDAR不可用的区域，仅从光学图像提供全幅高度估计。

**地上生物量测绘。** 回归模型结合光谱波段、植被指数和纹理特征，预测每个像素的生物量密度，用于碳核算和森林管理。

**建筑物高度估计。** 在LiDAR衍生高度上训练的回归模型可以从广泛可用的航空或卫星图像预测建筑物高度，用于城市规划和灾害风险评估。

**土壤属性预测。** 可以从多光谱图像结合地形和气候变量估计土壤湿度、有机碳、pH值和养分浓度。

这些应用有一个共同的工作流程：将图像与更直接但地理范围有限的来源的参考测量值配对，训练回归模型，并将其应用于在没有直接测量值的情况下生成全幅预测。

## 回归架构

### 适应回归的分割模型

U-Net、UNet++、DeepLabV3+和FPN等编码器-解码器网络都可以通过最小的修改用于回归。关键的改变在于输出头：将多类别softmax替换为单个输出通道，并且没有激活（或者对于非负目标使用ReLU）。`geoai`包通过设置`classes=1`自动处理此问题。

### 回归的损失函数

选择正确的损失函数会影响训练稳定性和预测质量：

**均方误差（MSE）** 由于平方运算而严重惩罚大误差，使其对异常值敏感，但对于最小化最坏情况误差有效。

**平均绝对误差（MAE，L1损失）** 对异常值更鲁棒，但由于梯度恒定，在最优值附近会减慢收敛速度。

**Huber损失（平滑L1）** 对于小误差表现得像MSE，对于大误差表现得像MAE，提供了一个平衡的折衷方案。当目标数据可能包含噪声或极端值时，它是一个很好的默认选择。

`geoai`包通过`loss_type`参数支持所有这三种损失：`"mse"`、`"l1"`和`"huber"`。

### 输出激活和缩放

对于许多地理空间任务，目标变量具有已知的物理范围。您可以通过两种方式纳入这种领域知识：

1. **目标范围过滤**：在瓦片创建过程中，使用`target_min`和`target_max`将值裁剪到有效范围，确保干净的训练数据。
2. **预测后裁剪**：在推理过程中，使用`predict_raster`中的`clip_range`参数将输出值裁剪到有效范围。

在实践中，结合这两种方法效果很好。

## 安装

取消注释以下行以安装所需的包。

In [ ]:
# %pip install -U "geoai-py[extra]"

## 案例研究：从Landsat图像预测NDVI

本案例研究训练了一个模型，用于从Landsat图像中预测像素级的NDVI。该模型在田纳西州诺克斯维尔的2022年数据上进行训练，然后应用于2023年图像以演示时间泛化。

### 环境设置

In [ ]:
import geoai
from sklearn.model_selection import train_test_split

### 下载数据

下载2022年（训练）和2023年（测试）的Landsat图像和NDVI栅格数据。

In [ ]:
train_raster = geoai.download_file(
    "https://data.source.coop/opengeos/geoai/tn_landsat_2022.tif"
)
train_target = geoai.download_file(
    "https://data.source.coop/opengeos/geoai/tn_ndvi_2022.tif"
)
test_raster = geoai.download_file(
    "https://data.source.coop/opengeos/geoai/tn_landsat_2023.tif"
)

### 检查数据

检查输入和目标栅格，以了解数据维度和值范围。

In [ ]:
import rasterio

with rasterio.open(train_raster) as src:
    in_channels = src.count
    print(f"Input shape: {src.height} x {src.width}, {src.count} bands")
    print(f"Input CRS: {src.crs}")
    print(f"Input resolution: {src.res}")

with rasterio.open(train_target) as src:
    print(f"Target shape: {src.height} x {src.width}, {src.count} band(s)")
    target_data = src.read(1)
    print(f"Target value range: [{target_data.min():.2f}, {target_data.max():.2f}]")

### 创建训练瓦片

将大型栅格瓦片化为较小的补丁进行训练。NDVI值被裁剪到[-1, 1]，过滤掉包含过多无数据像素的瓦片，并且50%的重叠（`stride=128`，`tile_size=256`）增加了训练样本。

In [ ]:
image_paths, target_paths = geoai.create_regression_tiles(
    input_raster=train_raster,
    target_raster=train_target,
    output_dir="ndvi_tiles",
    tile_size=256,
    stride=128,
    target_band=1,
    min_valid_ratio=0.9,
    target_min=-1.0,
    target_max=1.0,
)
print(f"Created {len(image_paths)} tiles")

### 拆分数据

将瓦片分为80%的训练集和20%的验证集。

In [ ]:
train_imgs, val_imgs, train_tgts, val_tgts = train_test_split(
    image_paths, target_paths, test_size=0.2, random_state=42
)
print(f"Training: {len(train_imgs)}, Validation: {len(val_imgs)}")

### 训练模型

训练一个带有ResNet34编码器的U-Net模型，用于像素级NDVI回归。该函数处理模型创建、数据加载、使用AdamW优化器进行训练、学习率调度和早期停止。

In [ ]:
model = geoai.train_pixel_regressor(
    train_image_paths=train_imgs,
    train_target_paths=train_tgts,
    val_image_paths=val_imgs,
    val_target_paths=val_tgts,
    encoder_name="resnet34",
    architecture="unet",
    in_channels=in_channels,
    output_dir="ndvi_model",
    batch_size=8,
    num_epochs=100,
    learning_rate=1e-3,
    num_workers=0,
    loss_type="mse",
    patience=20,
    devices=1,
    verbose=False,
)

### 监控训练历史

绘制训练和验证损失以及R平方曲线以诊断模型行为。两个损失都应同时下降，R平方应增加到1.0。

In [ ]:
fig, history_df = geoai.plot_training_history(
    log_dir="ndvi_model",
    metrics=["loss", "r2"],
)

### 在训练区域上运行推理

对2022年栅格运行推理以评估与地面真实情况的对比。该函数使用带有重叠的滑动窗口和高斯加权混合进行无缝预测。

In [ ]:
geoai.predict_raster(
    model=model,
    input_raster=train_raster,
    output_raster="ndvi_model/predicted_ndvi_2022.tif",
    tile_size=256,
    overlap=64,
    batch_size=8,
    clip_range=(-1.0, 1.0),
)

### 评估结果

比较预测的NDVI与地面真实情况。该函数并排生成地面真实情况、预测和残差图。

In [ ]:
fig, metrics = geoai.plot_regression_comparison(
    true_raster=train_target,
    pred_raster="ndvi_model/predicted_ndvi_2022.tif",
    title="NDVI Prediction Results",
    cmap="RdYlGn",
    vmin=-0.2,
    vmax=0.8,
    valid_range=(-1.0, 1.0),
)

创建预测值与实际值的散点图，以揭示系统偏差。点应聚集在1:1线上。

In [ ]:
fig, metrics = geoai.plot_scatter(
    true_raster=train_target,
    pred_raster="ndvi_model/predicted_ndvi_2022.tif",
    sample_size=50000,
    valid_range=(-1.0, 1.0),
    fit_line=True,
)

### 预测新数据（2023）

将训练好的模型应用于2023年的Landsat图像，以演示时间泛化。

In [ ]:
geoai.predict_raster(
    model=model,
    input_raster=test_raster,
    output_raster="ndvi_model/predicted_ndvi_2023.tif",
    tile_size=256,
    overlap=64,
    batch_size=8,
    clip_range=(-1.0, 1.0),
)

可视化2023年输入图像以及预测的NDVI图。

In [ ]:
geoai.visualize_prediction(
    input_raster=test_raster,
    pred_raster="ndvi_model/predicted_ndvi_2023.tif",
    cmap="RdYlGn",
    vmin=-0.2,
    vmax=0.8,
)

## 评估指标

回归模型使用与分类模型不同的指标：

**均方根误差（RMSE）** 以与目标变量相同的单位测量预测误差的标准差。

**平均绝对误差（MAE）** 是平均绝对预测误差，对异常值不如RMSE敏感。

**R平方** 衡量模型解释的方差比例，值接近1.0表示性能强劲。

**皮尔逊相关系数** 衡量预测值和目标值之间的线性关系，即使存在系统偏差，也能捕获模型是否正确地对像素进行排名。

检查残差图可以揭示模型误差的地理模式，例如水体附近或阴影区域的误差较高。

## 关键要点

1. **像素级回归** 为每个像素预测连续值，不同于分类和分割分配离散标签。

2. 用于分割的**编码器-解码器架构** 通过将输出更改为单个通道并切换损失函数，直接转换为回归。

3. **损失函数选择很重要**：MSE惩罚大误差，MAE对异常值鲁棒，Huber损失提供平衡的折衷方案。

4. **数据质量至关重要**：在瓦片化期间使用`target_min`/`target_max`，并在推理期间使用`clip_range`来强制有效值范围。

5. **`geoai`包** 通过`create_regression_tiles`、`train_pixel_regressor`、`predict_raster`和评估函数提供简化的工作流程。

6. **评估使用回归特定指标**：RMSE、MAE、R平方和相关性，辅以残差图和散点图。

7. **带有混合的重叠瓦片** 生成无缝预测图，没有可见的瓦片边界伪影。

8. **时间泛化** 是可能的，当模型学习到稳健的特征到目标的映射时，可以实现时间序列分析和间隙填充。